# 2일차 · 생산·품질 정보를 확인하는 웹 서비스

Let’s Grow with LG Display

**ChatGPT에 요청 → 전체 코드 복사 → Colab 코드 셀에 붙여 실행 → 웹 화면에서 확인**

오늘은 라인·LOT 조회, 불량 유형 비교, 검사 결과 입력·저장과 3D 설비 상태를 확인합니다. 모든 기록은 교육용 가상 데이터입니다.

Colab의 CPU 런타임을 연결합니다. Three.js/WebGL 화면은 내 브라우저에서 표시합니다. ChatGPT 유료 플랜을 사용하며 별도 LLM API 키는 입력하지 않습니다.

## 1.1 업무를 화면 요구로 바꾸기

```text
LG디스플레이 직무 이해용 생산·품질 조회 화면을 기획합니다.
사용자는 라인별 검사 결과를 확인하는 품질 담당자입니다.
라인 A/B와 LOT를 선택하면 생산량·달성률·검사 수량·불량률이 함께 바뀌어야 합니다.
불량 유형은 스크래치·얼룩·라인 불량입니다.
불량률의 계산식, 화면에 필요한 항목, 확인할 결과를 구체적으로 정리해 주세요.
설비의 가동 상태와 검사 기록을 구분해 설명해 주세요.
가상 데이터만 사용하며 실제 설비 고장 원인을 단정하지 마세요.
```

답변에서 선택 범위, 분모, 단위를 확인합니다.

**내가 정한 업무 질문:**

**사용자와 필요한 정보:**

## 1.2 가상 기록과 예상 값

| 라인·LOT | 목표 | 생산 | 검사 | 스크래치 | 얼룩 | 라인 불량 |
|---|---:|---:|---:|---:|---:|---:|
| A·A01 |500|400|100|1|1|0|
| A·A02 |500|400|100|0|1|1|
| B·B01 |400|300|100|3|2|0|
| B·B02 |400|350|100|0|0|1|

수량 단위는 개입니다. 불량은 세 유형의 합계입니다.

• 전체: 생산 1,450 / 검사 400 / 불량 10 / 불량률 2.50%
• 라인 A: 생산 800 / 검사 200 / 불량 4 / 불량률 2.00%
• 라인 B: 생산 650 / 검사 200 / 불량 6 / 불량률 3.00%
• B01: 생산 300 / 검사 100 / 불량 5 / 불량률 5.00%

생산 달성률 = 생산 합계 ÷ 목표 합계 × 100
불량률 = 불량 합계 ÷ 검사 합계 × 100
검사 0개이면 불량률은 **—**로 표시합니다.

## 2.1 ChatGPT에 조회 화면 요청

```text
LG디스플레이 생산·품질 조회 웹 서비스를 만들어 주세요.
첨부한 4개 LOT의 가상 데이터를 사용합니다. 라인 A/B/전체와 LOT를 선택하면 수치·기록 표·불량 유형 막대가 함께 바뀌게 하세요.
생산 달성률은 생산 합계÷목표 합계, 불량률은 불량 합계÷검사 합계로 계산하세요. 검사 0개이면 불량률은 —로 표시하세요.
Three.js로 금속 외장·롤러·유리 덮개·검사 헤드·상태등이 있는 패널 검사·이송 설비를 표현하세요. 회전·확대·가동·정지를 지원하세요.
선택 라인의 모의 설비를 표시하고 전체 조회에서는 라인 A를 표시하세요.
Colab용 Flask·Cloudflared 설치와 실행을 포함한 전체 Python 코드를 제공하세요. API 키는 사용하지 않습니다.

추가 실행 조건:
Colab Ubuntu에서 한 개의 Python 코드 셀로 실행합니다. Flask 설치·웹 서버·Cloudflared 임시 주소 출력을 포함하세요. Three.js와 OrbitControls는 동일한 고정 버전을 사용하세요. 통신은 async/await로 처리하세요. API 키 없이 실행하며 셀 재실행 시 이 실습의 기존 서버·터널만 종료하세요. 생략 없는 전체 Python 코드를 제공하세요.
가상 데이터는 아래 JSON과 같습니다. 한 패널에는 대표 불량 유형 한 가지를 기록합니다. 수치의 집계는 서버에서 수행하세요.
[
  {
    "line": "A",
    "lot": "A01",
    "target": 500,
    "produced": 400,
    "inspected": 100,
    "scratch": 1,
    "spot": 1,
    "line_defect": 0
  },
  {
    "line": "A",
    "lot": "A02",
    "target": 500,
    "produced": 400,
    "inspected": 100,
    "scratch": 0,
    "spot": 1,
    "line_defect": 1
  },
  {
    "line": "B",
    "lot": "B01",
    "target": 400,
    "produced": 300,
    "inspected": 100,
    "scratch": 3,
    "spot": 2,
    "line_defect": 0
  },
  {
    "line": "B",
    "lot": "B02",
    "target": 400,
    "produced": 350,
    "inspected": 100,
    "scratch": 0,
    "spot": 0,
    "line_defect": 1
  }
]
```

## 2.2 생성된 전체 코드 실행

아래 셀에 코드 전체를 붙이고 실행합니다. 가장 최근의 Cloudflared 주소를 새 탭으로 엽니다.

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 2.3 첫 실행 확인

전체 라인·전체 LOT에서 생산 1,450개, 달성률 80.6%, 불량률 2.50%를 확인합니다.
3D의 금속 외장·롤러·유리 덮개·검사 헤드·상태등을 회전해 확인합니다.

| 확인 항목 | 실제 결과 |
|---|---|
| 최근 주소 열림 | |
| 전체 수치 일치 | |
| 3D 회전·확대 | |
| 가동·정지 | |

오류가 있으면 현재 전체 코드와 마지막 오류 출력을 ChatGPT에 보내고, 수정된 전체 코드로 교체합니다.

## 2.4 실행 기준 코드

조회·입력·저장·CSV·3D 기능을 포함한 완성 예제입니다. 직접 생성한 코드가 실행되지 않을 때 비교하거나 실습을 이어갈 기준으로 사용합니다.

같은 런타임에서 다시 실행하면 이 실습의 이전 서버와 터널을 정리하고 새 주소를 출력합니다. 기록 파일이 있으면 읽고, 없으면 가상 기록 4개로 시작합니다. 주소가 나온 뒤 코랩 런타임을 유지합니다.

In [ ]:
import asyncio
import importlib.util
import json
import os
from pathlib import Path
import re
import socket
import subprocess
import sys
import threading
import urllib.request
from IPython.display import display, HTML as DisplayHTML

# ChatGPT 수정 요청 시 HTML_PAGE와 기능을 포함한 셀 전체를 전달합니다.
HTML_PAGE = r'''<!doctype html><html lang="ko"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>LGD 생산·품질 조회</title><style>*{box-sizing:border-box}body{margin:0;background:#eff2f6;color:#18212e;font:16px system-ui,-apple-system,'Noto Sans CJK KR',sans-serif}main{max-width:1400px;margin:auto;padding:26px}header{display:flex;justify-content:space-between;align-items:center;gap:20px;margin-bottom:22px}h1{font-size:30px;margin:7px 0}h2{font-size:20px;margin:0}small{color:#a50034;font-weight:700;letter-spacing:1px}.muted{color:#5b6573;line-height:1.6;margin:5px 0}.badge{background:#fff;padding:12px 17px;border-radius:8px;white-space:nowrap;font-weight:700}.metrics{display:grid;grid-template-columns:repeat(4,1fr);gap:16px;margin-bottom:20px}.metric{background:#fff;padding:18px 22px;border-top:3px solid #a50034;border-radius:8px}.metric strong{display:block;font-size:32px;margin:9px 0}.metric label{color:#5b6573;font-size:14px}.grid{display:grid;grid-template-columns:minmax(0,2.15fr) minmax(280px,1fr);gap:20px}.panel{background:#fff;border:1px solid #dbe1e8;border-radius:10px;overflow:hidden}.panelHead{padding:18px 20px;border-bottom:1px solid #e4e8ed;display:flex;align-items:center;justify-content:space-between}#viewport{height:475px;position:relative;background:#172331}#viewport canvas{display:block;touch-action:none;width:100%;height:100%}.hint{font-size:13px;line-height:1.5;background:#eef3f7;padding:12px 20px;color:#586575}#webglMessage{position:absolute;top:15px;left:15px;right:15px;color:#fff;background:#29425bdc;padding:12px;border-radius:6px;font-size:14px;pointer-events:none}.controls{padding:20px}button{font:inherit;font-weight:700;cursor:pointer;border:1px solid #ccd4de;background:#fff;color:#203145;border-radius:6px;padding:12px 15px}button:disabled{opacity:.5;cursor:wait}.primary{background:#a50034;color:#fff;border-color:#a50034}.secondary{background:#164c66;color:#fff;border-color:#164c66}.buttons{display:flex;gap:9px;flex-wrap:wrap;margin:16px 0}.status{font-size:24px;font-weight:750;color:#136749;margin:16px 0}.list{display:grid;grid-template-columns:1fr 1fr;gap:12px;border-top:1px solid #eee;padding-top:16px}.list dt{color:#6a7483}.list dd{margin:0;text-align:right;font-weight:700}.log{padding:12px;background:#f3f6f9;border-radius:6px;font-size:14px;line-height:1.7;min-height:72px;margin-top:16px}footer{font-size:13px;color:#687384;margin-top:16px;line-height:1.5}#connection{font-size:13px;color:#536b7b}#reset{padding:7px 10px;font-size:13px}@media(max-width:900px){.grid{grid-template-columns:1fr}.metrics{grid-template-columns:repeat(2,1fr)}h1{font-size:25px}header{align-items:flex-start}.badge{font-size:12px}#viewport{height:390px}}</style><style>.filters{display:flex;gap:18px;align-items:end;background:white;padding:16px;margin-bottom:16px;border-radius:8px}.filters label{display:grid;gap:7px}select,input{font:inherit;padding:10px;border:1px solid #bbc7d4;border-radius:5px;min-width:130px}#viewport{height:390px}.quality{padding:18px}.barrow{display:grid;grid-template-columns:70px 1fr 55px;gap:10px;align-items:center;margin:15px 0}.track{background:#edf0f5;height:18px}.bar{height:18px;background:#a50034}.lower{margin-top:20px}table{width:100%;border-collapse:collapse}th,td{text-align:left;padding:12px;border-bottom:1px solid #e0e6ed}th{background:#f1f4f7}.scroll{overflow:auto}.formgrid{display:grid;grid-template-columns:repeat(4,1fr);gap:14px;padding:18px}.formgrid label{display:grid;gap:6px}.formgrid input,.formgrid select{width:100%;min-width:0}summary{cursor:pointer;padding:18px;font-weight:bold}#formMessage{padding:0 18px 18px;white-space:pre-wrap}.toolbar{padding:18px;display:flex;gap:15px;align-items:center}a.download{color:#a50034;font-weight:bold}#viewLabel{font-weight:bold;color:#a50034}.status{margin:10px 0}.controls{padding:16px}.source{font-size:13px;line-height:1.5;color:#687384}#equipmentLabel{font-size:14px} @media(max-width:800px){.filters{flex-wrap:wrap}.formgrid{grid-template-columns:repeat(2,1fr)}} </style><script type="importmap">{"imports":{"three":"/static/vendor/three.module.js","three/addons/":"/static/vendor/"}}</script></head><body><main>
<header><div><small>LET’S GROW WITH LG DISPLAY · DAY 02</small><h1>생산·품질을 확인하는 웹 서비스</h1><p class="muted">라인과 LOT를 선택하고 검사 결과를 비교합니다.</p></div><div class="badge">교육용 가상 데이터</div></header>
<div class="filters"><label>생산라인<select id="line"><option value="all">전체 라인</option><option value="A">라인 A</option><option value="B">라인 B</option></select></label><label>LOT<select id="lot"><option value="all">전체 LOT</option></select></label><div><span id="viewLabel">조회 중</span><p class="source" id="updated"></p></div><a class="download" id="export" href="/api/export">조회 결과 CSV</a></div>
<section class="metrics"><div class="metric">목표 생산량<strong id="target">—</strong><label>선택 범위 합계 · 개</label></div><div class="metric">현재 생산량<strong id="produced">—</strong><label>선택 범위 합계 · 개</label></div><div class="metric">생산 달성률<strong id="achievement">—</strong><label>현재 합계 ÷ 목표 합계 × 100</label></div><div class="metric">불량률<strong id="defectRate">—</strong><label>불량 합계 ÷ 검사 합계 × 100</label></div></section>
<div class="grid"><section class="panel"><div class="panelHead"><h2>패널 검사·이송 설비</h2><button id="reset">시점 초기화</button></div><div id="viewport"><div id="webglMessage">3D 설비를 불러오는 중입니다.</div></div><div class="hint">드래그: 회전 · 휠: 확대/축소 · 가동/정지: 모의 이송 상태 변경</div></section><section class="panel"><div class="panelHead"><h2>설비와 검사 현황</h2><span id="connection">연결 확인 중</span></div><div class="controls"><div id="equipmentLabel">—</div><div class="status" id="status">—</div><div class="buttons"><button class="secondary" id="start">가동</button><button id="stop">정지</button><button id="check">현황 확인</button></div><dl class="list"><dt>검사 수량</dt><dd id="inspected">—</dd><dt>불량 수량</dt><dd id="defects">—</dd><dt>정상 수량</dt><dd id="good">—</dd></dl><div class="log" id="result">조회 결과를 확인합니다.</div><p class="source">3D는 선택 라인의 모의 상태를 표시합니다. 전체 조회에서는 라인 A를 표시합니다. 이송 동작은 검사 실적을 자동으로 늘리지 않습니다.</p></div></section></div>
<section class="panel lower" id="quality"><div class="panelHead"><h2>불량 유형별 수량</h2><span class="source">불량 패널 1개당 대표 유형 1개</span></div><div class="quality" id="bars"></div></section>
<section class="panel lower" id="recordsPanel"><div class="panelHead"><h2>LOT별 검사 기록</h2><span id="rowCount"></span></div><div class="scroll"><table><thead><tr><th>라인</th><th>LOT</th><th>목표</th><th>생산</th><th>검사</th><th>불량</th><th>불량률</th></tr></thead><tbody id="rows"></tbody></table></div></section>
<details class="panel lower" id="entry"><summary>새 LOT 검사 결과 입력</summary><form id="recordForm"><div class="formgrid"><label>생산라인<select name="line"><option>A</option><option>B</option></select></label><label>LOT<input name="lot" required maxlength="24" placeholder="A03"></label><label>목표 생산량<input name="target" type="number" min="1" max="100000" step="1" required value="500"></label><label>현재 생산량<input name="produced" type="number" min="0" max="100000" step="1" required value="420"></label><label>검사 수량<input name="inspected" type="number" min="0" max="100000" step="1" required value="100"></label><label>스크래치<input name="scratch" type="number" min="0" max="100000" step="1" required value="2"></label><label>얼룩<input name="spot" type="number" min="0" max="100000" step="1" required value="1"></label><label>라인 불량<input name="line_defect" type="number" min="0" max="100000" step="1" required value="0"></label></div><div class="toolbar"><button class="primary" id="save" type="submit">검사 결과 저장</button><span class="source">라인·LOT 조합은 한 번만 등록합니다. 저장 뒤 해당 라인의 전체 LOT를 조회합니다.</span></div><div id="formMessage" role="status"></div></form></details>
<footer>LG디스플레이 직무 이해용 모의 설비·가상 기록입니다. 실제 장비 사양이나 실제 검사 판정 기준을 재현하지 않습니다.<br>검사 기록은 현재 Colab의 /content/02_lgd_web/02_records.json에 저장합니다. 런타임 삭제 전 CSV와 JSON을 내려받습니다.</footer></main><script>
window.lgdState={running:false};const el=id=>document.getElementById(id);let serial=0;
function query(){return new URLSearchParams({line:el('line').value,lot:el('lot').value}).toString()}
function rate(a,b,d=2){return b===0?'—':(a/b*100).toFixed(d)+'%'}
function show(s){window.lgdState={running:s.running};for(const k of ['target','produced'])el(k).textContent=s[k].toLocaleString();el('achievement').textContent=rate(s.produced,s.target,1);el('defectRate').textContent=rate(s.defects,s.inspected);for(const k of ['inspected','defects'])el(k).textContent=s[k]+'개';el('good').textContent=(s.inspected-s.defects)+'개';el('status').textContent=s.running?'가동':'정지';el('status').style.color=s.running?'#136749':'#a50034';el('equipmentLabel').textContent='3D 표시·조작 대상: 라인 '+s.equipment_line;el('connection').textContent='서버 연결됨';el('viewLabel').textContent=(s.filter.line==='all'?'전체 라인':'라인 '+s.filter.line)+' · '+(s.filter.lot==='all'?'전체 LOT':s.filter.lot);el('updated').textContent='마지막 조회 '+new Date().toLocaleTimeString();el('export').href='/api/export?'+query();el('rowCount').textContent=s.rows.length+'건';
const old=el('lot').value;el('lot').replaceChildren(new Option('전체 LOT','all'));for(const lot of s.lots)el('lot').add(new Option(lot,lot));if(s.lots.includes(old))el('lot').value=old;
el('rows').replaceChildren();for(const r of s.rows){const tr=document.createElement('tr');for(const v of [r.line,r.lot,r.target,r.produced,r.inspected,r.defects,rate(r.defects,r.inspected)]){const td=document.createElement('td');td.textContent=v;tr.appendChild(td)}el('rows').appendChild(tr)}if(!s.rows.length){const tr=document.createElement('tr'),td=document.createElement('td');td.colSpan=7;td.textContent='조회 결과가 없습니다.';tr.appendChild(td);el('rows').appendChild(tr)}
el('bars').replaceChildren();const max=Math.max(1,...Object.values(s.types));for(const [k,label] of [['scratch','스크래치'],['spot','얼룩'],['line_defect','라인 불량']]){const row=document.createElement('div');row.className='barrow';const name=document.createElement('span');name.textContent=label;const track=document.createElement('div');track.className='track';const bar=document.createElement('div');bar.className='bar';bar.style.width=s.types[k]/max*100+'%';track.appendChild(bar);const value=document.createElement('strong');value.textContent=s.types[k]+'개';row.append(name,track,value);el('bars').appendChild(row)}}
async function load(){const ticket=++serial;try{const r=await fetch('/api/state?'+query(),{cache:'no-store'});const s=await r.json();if(!r.ok)throw Error(s.error||r.status);if(ticket===serial)show(s)}catch(e){if(ticket===serial){window.lgdState.running=false;el('connection').textContent='연결 끊김';el('result').textContent='조회 실패: '+e.message}}}
for(const id of ['line','lot'])el(id).addEventListener('change',async()=>{if(id==='line')el('lot').value='all';await load()});
for(const action of ['start','stop','check'])el(action).addEventListener('click',async()=>{for(const id of ['start','stop','check'])el(id).disabled=true;try{const line=el('line').value==='all'?'A':el('line').value;const r=await fetch('/api/action',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({action,line})});const s=await r.json();if(!r.ok)throw Error(s.error);await load();el('result').textContent=action==='check'?'현황을 확인했습니다.':'라인 '+line+' '+(s.running?'가동':'정지')+' 명령이 반영되었습니다.'}catch(e){el('result').textContent='명령 실패: '+e.message}finally{for(const id of ['start','stop','check'])el(id).disabled=false}});
el('recordForm').addEventListener('submit',async e=>{e.preventDefault();el('save').disabled=true;el('formMessage').textContent='저장 중입니다.';try{const data=Object.fromEntries(new FormData(e.target));for(const k of ['target','produced','inspected','scratch','spot','line_defect'])data[k]=Number(data[k]);const r=await fetch('/api/records',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify(data)});const s=await r.json();if(!r.ok)throw Error(s.error);el('line').value=data.line;el('lot').value='all';await load();el('formMessage').textContent='저장 완료: '+data.line+' / '+data.lot+' · 해당 라인의 합계가 갱신되었습니다.'}catch(e){el('formMessage').textContent='저장하지 않았습니다: '+e.message}finally{el('save').disabled=false}});
async function poll(){await load();setTimeout(async()=>{await poll()},2500)}poll();
</script><script type="module">
async function init(){try{const THREE=await import('three');const {OrbitControls}=await import('three/addons/OrbitControls.js');const host=document.getElementById('viewport');const scene=new THREE.Scene();scene.background=new THREE.Color(0x172331);scene.fog=new THREE.Fog(0x172331,16,34);const camera=new THREE.PerspectiveCamera(38,1,.1,70);camera.position.set(8.5,6.1,9.2);const renderer=new THREE.WebGLRenderer({antialias:true});renderer.setPixelRatio(Math.min(devicePixelRatio,2));renderer.shadowMap.enabled=true;renderer.shadowMap.type=THREE.PCFSoftShadowMap;renderer.toneMapping=THREE.ACESFilmicToneMapping;renderer.toneMappingExposure=1.25;host.appendChild(renderer.domElement);const controls=new OrbitControls(camera,renderer.domElement);controls.target.set(0,1.5,0);controls.enableDamping=true;controls.minDistance=5;controls.maxDistance=18;controls.maxPolarAngle=Math.PI/2.03;controls.update();document.getElementById('reset').addEventListener('click',async()=>{camera.position.set(8.5,6.1,9.2);controls.target.set(0,1.5,0);controls.update();});
scene.add(new THREE.HemisphereLight(0xdceeff,0x3c424c,2));const key=new THREE.DirectionalLight(0xffffff,3.3);key.position.set(4,10,7);key.castShadow=true;key.shadow.mapSize.set(2048,2048);key.shadow.camera.left=-8;key.shadow.camera.right=8;key.shadow.camera.top=8;key.shadow.camera.bottom=-8;scene.add(key);const fill=new THREE.DirectionalLight(0x72baff,1.7);fill.position.set(-6,4,-5);scene.add(fill);
const mat=(color,metalness=.25,roughness=.4)=>new THREE.MeshStandardMaterial({color,metalness,roughness});const shell=mat(0xdbe2e9,.45,.28),steel=mat(0x8d9ba8,.85,.25),dark=mat(0x182833,.4,.35),blue=mat(0x17647c,.65,.28),belt=mat(0x343d48,.2,.65),black=mat(0x121a22,.6,.25);function box(w,h,d,x,y,z,m,parent=scene){const mesh=new THREE.Mesh(new THREE.BoxGeometry(w,h,d),m);mesh.position.set(x,y,z);mesh.castShadow=true;mesh.receiveShadow=true;parent.add(mesh);return mesh;}function cyl(rad,len,x,y,z,m,rotation=0,parent=scene){const mesh=new THREE.Mesh(new THREE.CylinderGeometry(rad,rad,len,24),m);mesh.position.set(x,y,z);mesh.rotation.x=rotation;mesh.castShadow=true;mesh.receiveShadow=true;parent.add(mesh);return mesh;}
const floor=box(60,.1,60,0,-.15,0,mat(0x263442,.25,.7));const grid=new THREE.GridHelper(40,40,0x52606d,0x374652);grid.position.y=-.09;scene.add(grid);
// Cabinet base, adjustable feet, door seams, vents and fasteners.
box(5.2,.93,2.2,0,.72,0,shell);box(5.35,.16,2.34,0,1.22,0,blue);for(const x of [-2.2,2.2])for(const z of [-.87,.87]){cyl(.11,.38,x,.13,z,steel);cyl(.19,.07,x,-.02,z,dark);}for(const x of [-1.72,0,1.72]){box(1.67,.72,.025,x,.74,1.115,shell);box(.045,.22,.06,x+.58,.8,1.15,steel);}for(let i=0;i<9;i++)box(.32,.021,.025,-2.0,.49+i*.055,1.145,dark);for(const x of [-2.48,2.48])for(const y of [.4,1.05]){const b=cyl(.035,.035,x,y,1.15,steel,Math.PI/2);}
// Long roller conveyor and guide rails.
box(7.0,.13,1.47,0,1.43,0,belt);for(let i=0;i<26;i++)cyl(.065,1.34,-3.28+i*.26,1.53,0,steel,Math.PI/2);for(const z of [-.82,.82]){box(7.1,.17,.11,0,1.6,z,steel);box(7.1,.055,.13,0,1.72,z,blue);}for(const x of [-3.05,3.05])for(const z of [-.68,.68])box(.12,1.2,.12,x,.72,z,steel);
// Inspection enclosure, transparent panels, lintel and inspection camera.
for(const x of [-1.55,1.55])for(const z of [-1,1])box(.13,1.78,.13,x,2.2,z,steel);box(3.28,.22,2.18,0,3.12,0,shell);box(3.28,.12,2.18,0,3.26,0,blue);const glass=new THREE.MeshPhysicalMaterial({color:0x70b9cc,transparent:true,opacity:.17,metalness:.05,roughness:.13,depthWrite:false});box(3.0,1.32,.018,0,2.32,-1,glass);box(3,1.3,.018,0,2.33,1,glass);box(.04,1.32,.035,0,2.32,1.02,steel);box(.05,.3,.065,.16,2.28,1.075,dark);box(2.55,.11,.15,0,2.82,0,steel);box(.7,.3,.48,0,2.58,0,black);cyl(.16,.16,0,2.35,0,black);cyl(.12,.025,0,2.25,0,new THREE.MeshStandardMaterial({color:0x72e7ff,emissive:0x26c8ff,emissiveIntensity:1.5}));const scan=box(.026,.012,1.10,0,1.66,0,new THREE.MeshBasicMaterial({color:0x65e5ff,transparent:true,opacity:.65}));
// Operator station and tower light.
box(.13,.92,.13,2.1,1.72,1.05,steel);box(.73,.51,.12,2.1,2.22,1.03,dark);box(.61,.36,.024,2.1,2.24,1.106,new THREE.MeshStandardMaterial({color:0x177d97,emissive:0x075169,emissiveIntensity:.8}));for(let i=0;i<3;i++)box(.1,.025,.012,1.92+i*.18,2.3,1.123,mat(0x8aeaff));cyl(.045,.42,1.25,3.51,-.75,steel);const green=new THREE.MeshStandardMaterial({color:0x146b48,emissive:0x16c97c,emissiveIntensity:1});const red=new THREE.MeshStandardMaterial({color:0xc72447,emissive:0xf52f51,emissiveIntensity:0});cyl(.09,.15,1.25,3.66,-.75,green);cyl(.09,.15,1.25,3.83,-.75,mat(0xa78420));cyl(.09,.15,1.25,4,-.75,red);cyl(.1,.04,1.25,4.1,-.75,black);
// Display panels use reflective glass-like surfaces; motion is illustrative.
const panels=[];for(let i=0;i<3;i++){const g=new THREE.Group();box(.98,.055,1.04,0,0,0,dark,g);box(.88,.012,.94,0,.035,0,new THREE.MeshPhysicalMaterial({color:0x163d5a,metalness:.8,roughness:.16,clearcoat:1}),g);box(.018,.014,.9,-.31,.045,0,blue,g);g.position.set(-2.8+i*2.5,1.63,0);scene.add(g);panels.push(g);}const clock=new THREE.Clock();function resize(){const w=host.clientWidth,h=host.clientHeight;renderer.setSize(w,h);camera.aspect=w/h;camera.updateProjectionMatrix();}new ResizeObserver(resize).observe(host);resize();function animate(){requestAnimationFrame(animate);const dt=Math.min(clock.getDelta(),.05);const running=window.lgdState.running;for(const panel of panels){if(running){panel.position.x+=dt*.6;if(panel.position.x>3.6)panel.position.x=-3.6;}}green.emissiveIntensity=running?1.7:0;red.emissiveIntensity=running?0:.9;scan.visible=running;controls.update();renderer.render(scene,camera);}animate();document.getElementById('webglMessage').style.display='none';window.lgd3DReady=true;}catch(error){document.getElementById('webglMessage').textContent='3D 표시를 확인해 주세요. 브라우저의 WebGL·하드웨어 가속 또는 새로고침을 확인합니다. '+error.message;window.lgd3DError=String(error);}}await init();
</script></body></html>
'''

async def _lgd_download(url, destination):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 1000:
        return
    def transfer():
        req = urllib.request.Request(url, headers={"User-Agent": "LGD-Education/1.0"})
        with urllib.request.urlopen(req, timeout=60) as response:
            content = response.read()
        if len(content) < 1000:
            raise RuntimeError("다운로드 결과가 너무 작습니다: " + destination.name)
        temporary = destination.with_suffix(destination.suffix + ".tmp")
        temporary.write_bytes(content)
        temporary.replace(destination)
    for attempt in range(3):
        try:
            await asyncio.to_thread(transfer)
            return
        except Exception:
            if attempt == 2:
                raise
            await asyncio.sleep(2)

async def _lgd_start():
    global _LGD_DAY2_SERVER, _LGD_DAY2_TUNNEL, _LGD_DAY2_LOG
    if importlib.util.find_spec("flask") is None:
        print("웹 서버 구성요소를 설치합니다.", flush=True)
        await asyncio.to_thread(subprocess.check_call, [sys.executable, "-m", "pip", "-q", "install", "flask==3.1.2"])
        import site
        site.addsitedir(site.getusersitepackages())
        importlib.invalidate_caches()
    from flask import Flask, jsonify, request, make_response
    from werkzeug.serving import make_server
    import logging
    logging.getLogger("werkzeug").setLevel(logging.ERROR)
    root = Path(os.environ.get("LGD_DAY2_ROOT", "/content/02_lgd_web"))
    vendor = root / "static" / "vendor"
    vendor.mkdir(parents=True, exist_ok=True)
    (root / "02_dashboard.html").write_text(HTML_PAGE, encoding="utf-8")
    print("1/3 Three.js 구성요소를 준비합니다.", flush=True)
    await _lgd_download("https://cdn.jsdelivr.net/npm/three@0.170.0/build/three.module.js", vendor / "three.module.js")
    await _lgd_download("https://cdn.jsdelivr.net/npm/three@0.170.0/examples/jsm/controls/OrbitControls.js", vendor / "OrbitControls.js")
    # 이 셀이 생성한 서버와 터널만 종료한 뒤 재실행합니다.
    old_tunnel = globals().get("_LGD_DAY2_TUNNEL")
    if old_tunnel and old_tunnel.poll() is None:
        old_tunnel.terminate()
        try:
            await asyncio.to_thread(old_tunnel.wait, 5)
        except subprocess.TimeoutExpired:
            old_tunnel.kill()
            await asyncio.to_thread(old_tunnel.wait, 5)
    old_log = globals().get("_LGD_DAY2_LOG")
    if old_log:
        old_log.close()
    old_server = globals().get("_LGD_DAY2_SERVER")
    if old_server:
        await asyncio.to_thread(old_server.shutdown)
        old_server.server_close()
    # 다른 실습이 쓰는 포트는 건드리지 않고 빈 포트를 선택합니다.
    port = None
    for candidate in range(8000, 8021):
        with socket.socket() as probe:
            try:
                probe.bind(("127.0.0.1", candidate))
                port = candidate
                break
            except OSError:
                continue
    if port is None:
        raise RuntimeError("사용 가능한 포트가 없습니다. 현재 런타임의 다른 실습을 확인해 주세요.")
    app = Flask("lgd_day2", static_folder=str(root / "static"))
    import csv, io, copy
    from datetime import datetime
    data_path = root / "02_records.json"
    seed = [{'line': 'A', 'lot': 'A01', 'target': 500, 'produced': 400, 'inspected': 100, 'scratch': 1, 'spot': 1, 'line_defect': 0}, {'line': 'A', 'lot': 'A02', 'target': 500, 'produced': 400, 'inspected': 100, 'scratch': 0, 'spot': 1, 'line_defect': 1}, {'line': 'B', 'lot': 'B01', 'target': 400, 'produced': 300, 'inspected': 100, 'scratch': 3, 'spot': 2, 'line_defect': 0}, {'line': 'B', 'lot': 'B02', 'target': 400, 'produced': 350, 'inspected': 100, 'scratch': 0, 'spot': 0, 'line_defect': 1}]
    if not data_path.exists():
        data_path.write_text(json.dumps(seed, ensure_ascii=False, indent=2), encoding="utf-8")
    records = json.loads(data_path.read_text(encoding="utf-8"))
    lock = threading.Lock()
    equipment = {"A": True, "B": True}
    def selected():
        line, lot = request.args.get("line", "all"), request.args.get("lot", "all")
        if line not in {"all", "A", "B"}:
            raise ValueError("생산라인을 확인해 주세요.")
        base = [r for r in records if line == "all" or r["line"] == line]
        rows = [dict(r, defects=r["scratch"]+r["spot"]+r["line_defect"]) for r in base if lot == "all" or r["lot"] == lot]
        totals = {k: sum(r[k] for r in rows) for k in ["target", "produced", "inspected", "defects"]}
        active = "A" if line == "all" else line
        return dict(**totals, rows=rows, types={k:sum(r[k] for r in rows) for k in ["scratch", "spot", "line_defect"]}, lots=sorted({r["lot"] for r in base}), filter=dict(line=line, lot=lot), equipment_line=active, running=equipment[active])
    @app.get("/")
    def page():
        response = make_response(HTML_PAGE)
        response.headers["Cache-Control"] = "no-store"
        return response
    @app.get("/health")
    def health():
        return jsonify(ok=True, project="LGD Day 2")
    @app.get("/api/state")
    def read_state():
        with lock:
            try:
                return jsonify(selected())
            except ValueError as exc:
                return jsonify(error=str(exc)), 400
    @app.post("/api/action")
    def action():
        data = request.get_json(silent=True)
        if not isinstance(data,dict) or data.get("line") not in equipment or data.get("action") not in {"start","stop","check"}:
            return jsonify(error="라인과 명령을 확인해 주세요."),400
        with lock:
            if data["action"] != "check":
                equipment[data["line"]] = data["action"] == "start"
            return jsonify(running=equipment[data["line"]])
    @app.post("/api/records")
    def add_record():
        data = request.get_json(silent=True)
        if not isinstance(data,dict):
            return jsonify(error="입력 형식을 확인해 주세요."),400
        line,lot=data.get("line"),data.get("lot")
        if line not in equipment or not isinstance(lot,str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,24}",lot):
            return jsonify(error="라인은 A/B, LOT는 영문·숫자·밑줄·하이픈 1~24자로 입력합니다."),400
        keys=["target","produced","inspected","scratch","spot","line_defect"]
        if any(type(data.get(k)) is not int or not 0 <= data[k] <= 100000 for k in keys) or data["target"] == 0:
            return jsonify(error="수량은 0~100000의 정수, 목표는 1 이상으로 입력합니다."),400
        if data["inspected"] > data["produced"]:
            return jsonify(error="검사 수량은 생산 수량을 초과할 수 없습니다."),400
        if sum(data[k] for k in ["scratch","spot","line_defect"]) > data["inspected"]:
            return jsonify(error="불량 유형 합계는 검사 수량을 초과할 수 없습니다."),400
        row={k:data[k] for k in keys};row.update(line=line,lot=lot)
        with lock:
            if any(r["line"]==line and r["lot"]==lot for r in records):
                return jsonify(error="이미 등록된 라인·LOT입니다. 새로운 LOT를 입력합니다."),409
            proposed=records+[row]
            try:
                temporary=data_path.with_suffix(".tmp")
                temporary.write_text(json.dumps(proposed,ensure_ascii=False,indent=2),encoding="utf-8")
                temporary.replace(data_path)
            except OSError:
                return jsonify(error="파일 저장에 실패했습니다. 기록은 추가하지 않았습니다."),500
            records.append(row)
        return jsonify(ok=True,record=row),201
    @app.get("/api/export")
    def export():
        with lock:
            try:
                rows=selected()["rows"]
            except ValueError as exc:
                return jsonify(error=str(exc)),400
        stream=io.StringIO();fields=["line","lot","target","produced","inspected","scratch","spot","line_defect","defects"]
        writer=csv.DictWriter(stream,fieldnames=fields);writer.writeheader();writer.writerows(rows)
        response=make_response("\ufeff"+stream.getvalue());response.headers["Content-Type"]="text/csv; charset=utf-8";response.headers["Content-Disposition"]='attachment; filename="02_LGD_Inspection_Results.csv"'
        return response
    _LGD_DAY2_SERVER = make_server("127.0.0.1", port, app, threaded=True)
    threading.Thread(target=_LGD_DAY2_SERVER.serve_forever, daemon=True).start()
    local_url = f"http://127.0.0.1:{port}"
    def check_health():
        with urllib.request.urlopen(local_url + "/health", timeout=10) as response:
            return json.load(response)
    assert (await asyncio.to_thread(check_health))["ok"]
    print("2/3 웹 서버가 실행되었습니다.", flush=True)
    if os.environ.get("LGD_DAY2_LOCAL_TEST") == "1":
        print(local_url)
        return local_url
    cloudflared = root / "cloudflared"
    await _lgd_download("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cloudflared)
    cloudflared.chmod(0o755)
    log_path = root / "02_cloudflared.log"
    _LGD_DAY2_LOG = log_path.open("w", encoding="utf-8")
    _LGD_DAY2_TUNNEL = subprocess.Popen(
        [str(cloudflared), "tunnel", "--url", local_url, "--protocol", "http2", "--no-autoupdate"],
        stdout=_LGD_DAY2_LOG, stderr=subprocess.STDOUT,
    )
    print("3/3 Cloudflared 접속 주소를 생성합니다. 잠시 기다려 주세요.", flush=True)
    public_url = None
    for _ in range(90):
        await asyncio.sleep(1)
        log_text = log_path.read_text(encoding="utf-8", errors="replace")
        match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log_text)
        if match:
            public_url = match.group(0)
            break
        if _LGD_DAY2_TUNNEL.poll() is not None:
            break
    if not public_url:
        print(log_path.read_text(encoding="utf-8", errors="replace")[-1800:])
        raise RuntimeError("접속 주소를 만들지 못했습니다. 마지막 오류 문구를 확인한 뒤 셀을 다시 실행해 주세요.")
    print("웹 화면 주소:", public_url)
    display(DisplayHTML(f'<p><a href="{public_url}" target="_blank" rel="noopener noreferrer" style="font-size:20px;font-weight:bold">디스플레이 실습 화면 열기</a></p>'))
    print("주소가 생성되어도 접속 준비에 잠시 시간이 걸릴 수 있습니다. 새 탭으로 열어 확인합니다.")
    print("Colab 런타임을 유지하세요. 다시 실행하면 주소가 바뀔 수 있습니다.")
    return public_url

await _lgd_start()


## 2.5 라인·LOT 필터 수정

현재 전체 코드와 아래 요청을 함께 전달합니다.

```text
라인을 바꾸면 LOT 선택을 “전체 LOT”로 되돌리고, 해당 라인의 LOT만 선택 목록에 표시하세요.
상단 수치·불량 유형 막대·기록 표·CSV가 동일한 조회 조건을 사용하게 하세요.
조회 범위를 화면에 “라인 B · B01”처럼 표시하세요.
결과가 없으면 “조회 결과가 없습니다.”를 표시하고 비율은 —로 표시하세요.
현재 Three.js 설비·가동/정지·서버·Cloudflared 기능을 유지하세요.
수정된 전체 Colab Python 코드를 제공하세요.
```

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 2.6 불량 유형 표시 개선

```text
불량 유형 막대 옆에 수량과 “선택 범위 내 유형별 수량”이라는 설명을 표시하세요.
유형별 합계가 상단 불량 수량과 일치하도록 같은 데이터를 사용하세요.
불량이 0개이면 막대는 0, 수량은 0개로 표시하세요.
검사 수량이 0이면 불량률을 0%로 단정하지 말고 —로 표시하세요.
조회·3D·가동/정지 기능을 유지하고 전체 Colab 코드를 제공하세요.
```

전체 → A → B → B01 → 전체를 선택하며 표·수치·막대가 함께 바뀌는지 확인합니다. B01의 유형별 수량은 3 / 2 / 0입니다.

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 2.7 오류 수정 요청

```text
현재 전체 코드: [코드 전체 붙이기]
실행 단계: [Colab 실행 / 주소 접속 / 필터 선택 / 저장]
기대한 결과: [선택 범위와 예상 값]
실제 결과: [화면 값 또는 오류 원문]
원인을 쉽게 설명하고 수정된 전체 Colab Python 코드를 제공하세요.
기존 조회·입력·저장·CSV·Three.js·Cloudflared 기능은 유지하세요.
부분 교체나 생략 없이 셀 전체를 제공하세요.
```

## 3.1 검사 결과 입력·저장 요청

```text
“새 LOT 검사 결과 입력” 폼을 추가하세요. 필드는 라인·LOT·목표·생산·검사·스크래치·얼룩·라인 불량입니다.
목표는 1 이상, 다른 수량은 0 이상의 정수로 제한하세요. 검사≤생산, 불량 유형 합계≤검사를 서버에서도 검증하세요.
라인·LOT가 이미 있으면 중복 저장을 거부하세요.
/content/02_lgd_web/02_records.json에 저장하고, 파일이 있을 때는 기존 기록을 읽으세요.
파일 저장 성공 후에만 완료 문구를 표시하고 해당 라인의 전체 LOT를 갱신하세요.
기존 조회·3D·Cloudflared 기능을 유지한 전체 Colab Python 코드를 제공하세요.
```

**새 기록:** 라인 A / LOT A03 / 목표 500 / 생산 420 / 검사 100 / 스크래치 2 / 얼룩 1 / 라인 불량 0

## 3.2 전체 코드 실행 및 저장 검증

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


### 저장 확인

“새 LOT 검사 결과 입력”을 열고 A03을 한 번 저장합니다.

• A 전체: 목표 1,500 / 생산 1,220 / 검사 300 / 불량 7 / 불량률 2.33%
• 전체 라인: 생산 1,870 / 검사 500 / 불량 13 / 불량률 2.60%
• 같은 A03을 다시 입력: 중복 안내, 행 수와 합계 유지
• 새 LOT에 검사 100, 스크래치 101: 저장 거부, 기존 합계 유지
• 새 LOT에 생산 50, 검사 100: 저장 거부
• 새 LOT에 검사 0, 모든 불량 0: 저장 허용, 해당 LOT의 불량률 —

검사 0개 시험용 LOT는 기본 값 비교를 마친 뒤 추가합니다. 새 기록을 추가하면 전체 합계가 바뀝니다.

**정상 저장 결과:**

**거부 사례와 실제 문구:**

## 3.3 CSV 내보내기 요청

```text
현재 선택한 라인·LOT의 기록만 CSV로 내려받는 “조회 결과 CSV” 기능을 추가하세요.
열은 line, lot, target, produced, inspected, scratch, spot, line_defect, defects입니다.
파일 이름은 02_LGD_Inspection_Results.csv로 하고 UTF-8 BOM을 포함하세요.
화면과 CSV가 같은 조회 조건과 원본 기록을 사용하게 하세요.
입력·저장·3D·서버·Cloudflared 기능을 유지한 전체 Colab 코드를 제공하세요.
```

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 3.4 화면과 CSV 대조

A / A03을 선택하고 “조회 결과 CSV”를 누릅니다.
CSV의 데이터 행은 1개, produced=420, inspected=100, defects=3인지 확인합니다.
전체를 선택해 다시 내려받으면 기본 4개와 A03을 합쳐 5개 행입니다. 추가 시험용 LOT가 있다면 그 수만큼 늘어납니다.

같은 런타임에서 기준 코드 셀을 다시 실행합니다. 새 주소에서 A03이 한 번만 남아 있는지 확인합니다.

**재실행 후 확인 결과:**

## 3.5 기록 파일 내려받기

다음 셀은 현재 기록 JSON과 전체 기록 CSV를 ZIP으로 내려받습니다.
노트북도 Colab의 파일 메뉴에서 .ipynb로 내려받아 함께 보관합니다. /content 파일은 런타임 삭제 시 사라질 수 있습니다.

In [ ]:
from pathlib import Path
import json, csv, io, zipfile
from google.colab import files

root = Path("/content/02_lgd_web")
source = root / "02_records.json"
if not source.exists():
    raise FileNotFoundError("먼저 웹 서비스 코드를 실행하고 검사 기록을 저장합니다.")
records = json.loads(source.read_text(encoding="utf-8"))
stream = io.StringIO()
fields = ["line", "lot", "target", "produced", "inspected", "scratch", "spot", "line_defect", "defects"]
writer = csv.DictWriter(stream, fieldnames=fields)
writer.writeheader()
for record in records:
    row = dict(record)
    row["defects"] = sum(row[k] for k in ["scratch", "spot", "line_defect"])
    writer.writerow({k: row[k] for k in fields})
archive = Path("/content/02_LGD_Inspection_Data.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    bundle.write(source, "02_records.json")
    bundle.writestr("02_LGD_Inspection_Results.csv", ("\ufeff" + stream.getvalue()).encode("utf-8"))
files.download(str(archive))


## 4.1 3D 설비 상태 요청

```text
3D 표시·조작 대상 라인을 화면에 명확히 표시하세요. 전체 조회에서는 라인 A를 사용하세요.
가동·정지 버튼은 해당 라인의 서버 상태를 바꾸고, 패널 이송·상태 문구·상태등이 이를 따르게 하세요.
라인 B를 정지한 뒤 A로 바꾸면 A의 상태를 표시하고, B로 돌아오면 정지가 유지되어야 합니다.
조회 실패 시 마지막 정보를 정상 연결로 표시하지 말고 연결 끊김을 알리세요.
검사 실적과 유형별 수량은 모의 이동으로 늘리지 마세요.
기존 기능을 유지한 전체 Colab Python 코드를 제공하세요.
```

## 4.2 전체 코드 실행 및 라인별 동작 확인

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


### 동작 확인

B 선택 → 정지 → A 선택 → A 가동 확인 → B 재선택 → B 정지 유지 확인 → B 가동

전체 조회에서는 3D 표시·조작 대상이 라인 A인지 확인합니다. 동작 전후 검사 실적이 달라지지 않아야 합니다.

**라인별 확인 결과:**

## 4.3 3D 설비 표현 개선

```text
Three.js 패널 검사·이송 설비의 표현을 개선하세요.
검사 헤드가 패널 위에 보이도록 하고, 패널이 롤러 위에서 이동하게 하세요.
금속 외장·프레임·유리 덮개·상태등의 재질 차이를 유지하세요.
패널이 외장이나 기둥을 관통하지 않게 하고, 카메라 초기 위치에서 이송 경로가 보이게 하세요.
마우스 회전·확대, 라인 선택·가동/정지, 조회·입력·저장 기능을 유지하세요.
전체 Colab Python 코드를 제공하세요.
```

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 4.4 팀의 서비스 개선

```text
우리 팀 사용자는 [생산/품질/설비 담당자]입니다.
사용자가 가장 먼저 확인할 질문은 [업무 질문]입니다.
현재 화면의 문제는 [관찰한 불편]입니다.
[바꿀 화면·동작 한 가지]를 개선하세요.
완료 기준은 [사용자가 수행할 행동과 확인할 값]입니다.
계산식·라인/LOT 조회·입력 검증·저장·CSV·3D 가동/정지 기능은 유지하세요.
전체 Colab Python 코드를 제공하세요.
```

**문제와 변경 이유:**

**완료 기준:**

## 4.5 전체 코드 실행 및 팀 간 확인

In [ ]:
# ChatGPT가 작성한 전체 Python 코드를 이 셀에 붙여 넣고 실행합니다.


## 4.6 결과 기록

| 항목 | 예상 | 실제 | 보완할 내용 |
|---|---|---|---|
| LOT 조회 | 범위·수치·표 일치 | | |
| 검사 결과 저장 | 새 행과 합계 갱신 | | |
| 중복·잘못된 입력 | 거부·기존 기록 유지 | | |
| CSV | 선택 범위와 동일 | | |
| 같은 런타임 재실행 | 기록 유지 | | |
| 라인별 3D 동작 | 선택 라인 상태 일치 | | |

**내가 작성한 요청 또는 확인한 결과:**

노트북, CSV, JSON, 캡처를 함께 보관합니다.

## 참고 자료

• [Google Colab FAQ](https://research.google.com/colaboratory/faq.html)
• [Cloudflare Quick Tunnels](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/)
• [Three.js 설치](https://threejs.org/manual/en/installation.html)

임시 터널은 실습 접속용입니다. Colab 런타임과 터널 상태에 따라 접속이 종료될 수 있습니다. 노트북에서 코드를 실행하고 결과를 확인하는 실습으로 사용합니다.